# Stage C — locked tokenizer stream dataset
Build the ANI/species-separated, contig-ordered, memory-mapped Stage C corpus after Handoff 00 selects a tokenizer. See `docs/titans_stage_c/handoffs/00b_stream_dataset.md`.


In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='b1261518148aed6ef07fa2f8b255407ef76fff93'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
# Add the shared source-dataset folder to My Drive as a shortcut; do not copy its 15 Gbp contents.
SOURCE_ROOT='/content/drive/MyDrive/bacteria_titan_v1_ecoli_related_15gbp'
OUTPUT_ROOT=f'{DRIVE_ROOT}/stage_c_dataset'
ANI_PAIRS=f'{DRIVE_ROOT}/inputs/ecoli_skani_triangle.tsv'
SKANI_BINARY='/content/bin/skani'
SKANI_RELEASE_URL='https://github.com/bluenote-1577/skani/releases/download/latest/skani'
SKANI_THREADS=4
REBUILD_ANI_PAIRS=False
TOKENIZER_SELECTION=f'{DRIVE_ROOT}/runs/c1_tokenizers_cpu/tokenizer_selection.json'
DNABERT2_PATH=f'{DRIVE_ROOT}/tokenizers/dnabert2'
BACTERIAL_BPE_PATH=f'{DRIVE_ROOT}/runs/c1_tokenizers_cpu/bacterial_train_only_bpe'
RUN_NAME='c2_stream_dataset'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess,sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
# Reinstall the CPU-pilot ABI stack; Colab can retain extensions built for NumPy 2.
subprocess.run([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--no-cache-dir','numpy==1.26.4','pandas==2.2.2','pyarrow==18.1.0'],check=True)
import numpy,pandas,pyarrow
print({'numpy':numpy.__version__,'pandas':pandas.__version__,'pyarrow':pyarrow.__version__})
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan,evo2-tokenizer]'],check=True)
required=[Path(SOURCE_ROOT)/'manifests'/'accession_manifest.parquet',Path(SOURCE_ROOT)/'raw'/'ncbi_dataset_zips',Path(TOKENIZER_SELECTION)]
missing=[str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError('Missing Stage C inputs: '+', '.join(missing))


In [ ]:
# Generate reproducible ANI99 evidence for the E. coli split.
import pandas as pd
import shutil
run_dir=f'{DRIVE_ROOT}/runs/{RUN_NAME}'
ani_pairs=Path(ANI_PAIRS)
if REBUILD_ANI_PAIRS or not ani_pairs.exists():
    resolved=shutil.which(SKANI_BINARY)
    skani=Path(resolved) if resolved else Path(SKANI_BINARY)
    if not skani.exists():
        skani.parent.mkdir(parents=True,exist_ok=True)
        subprocess.run(['curl','-L','--fail','--retry','3','-o',str(skani),SKANI_RELEASE_URL],check=True)
        skani.chmod(skani.stat().st_mode | 0o111)
    subprocess.run([str(skani),'triangle','-h'],check=True,stdout=subprocess.DEVNULL)
    command=[sys.executable,str(repo/'scripts/generate_stage_c_ani_pairs.py'),'--source-root',SOURCE_ROOT,'--output',ANI_PAIRS,'--work-dir','/content/stage_c_ecoli_skani_fastas','--threads',str(SKANI_THREADS),'--skani',str(skani)]
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',run_dir,'--label','generate_ani_pairs','--repo',str(repo),'--',*command],check=True)
else:
    print(f'Reusing existing ANI evidence: {ani_pairs}')
columns=set(pd.read_csv(ani_pairs,sep='\t',nrows=0).columns)
required_columns={'Ref_file','Query_file','ANI'}
if missing:=required_columns-columns: raise ValueError('ANI evidence missing columns: '+', '.join(sorted(missing)))
print(f'ANI evidence ready: {ani_pairs}')

run_dir=f'{DRIVE_ROOT}/runs/{RUN_NAME}'
command=[sys.executable,str(repo/'scripts/build_bacteria_titan_stage_c_streams.py'),'--source-root',SOURCE_ROOT,'--output-root',OUTPUT_ROOT,'--ani-pairs',ANI_PAIRS,'--tokenizer','auto','--tokenizer-selection',TOKENIZER_SELECTION,'--dnabert2-path',DNABERT2_PATH,'--bacterial-bpe-path',BACTERIAL_BPE_PATH]
subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',run_dir,'--label','build_stream_dataset','--repo',str(repo),'--',*command],check=True)


In [ ]:
import json
selection=json.loads(Path(TOKENIZER_SELECTION).read_text())
dataset=Path(OUTPUT_ROOT)/'ordered_streams'/selection['selected_tokenizer']
manifest=json.loads((dataset/'token_stream_manifest.json').read_text())
print(json.dumps({'dataset':str(dataset),'streams':manifest['streams'],'tokens':manifest['tokens'],'bases':manifest['bases'],'tokenizer':manifest['tokenizer']['name']},indent=2))
print('SHARE THESE DIRECTORIES:',run_dir,dataset)
